# 27 — Ligand-chem GBSA surrogate

Can a per-complex ML surrogate predict GBSA ΔG from cheap descriptors? Three tiers, each removing more leakage: quasi-circular → partial-leak → clean ligand-chemistry only. The clean tier is the deployable number, and it's modest.

**Method.** The 0.93 → 0.778 → 0.455 cascade is per-target Pearson r; n = 9 per correlation; bootstrap CIs use B = 1000. The cascade goes from quasi-circular (features include GBSA energy inputs) → partial-leak (MD stability retained) → ligand-chem-only.

> **Bootstrap regime.** This notebook uses B=1000, seed 20250901 for speed. The canonical repo-wide regime (see `data/derived/canonical_baselines.csv`) is B=5000, seed 20260902. Numbers here are stable at reported precision; CIs are a hair wider.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

> **Reader's guide — what this notebook does, in plain language**\n>\n> **Question:** Can we predict per-complex GBSA ΔG with a cheaper ML surrogate,\n> without running 30 ns MD each time? If yes → 1000× cheaper than testing every ligand.\n>\n> **Key concept: data leakage** — the most important ML lesson in this repo.\n>\n> If a feature *already tells you part of the answer*, your model looks brilliant\n> but is worthless on new data. Example: `lig_dipole_mean_eA` (dipole moment)\n> is a direct input to the MM-GBSA solvation term. A model predicting GBSA from\n> dipole isn't making a prediction — it's reconstructing a known formula.\n>\n> That's why we run **3 leakage tiers** and watch what happens:\n>\n> | Tier | Features | How \"deployable\"? | Median r |\n> |---|---|---|---:|\n> | **1 quasi-circular** | All 60 MD features (incl. dipole, SASA, VDW sums, salt bridges) | **Worthless** — GBSA inputs are in there | **0.93** |\n> | **2 partial-leak** | MD features minus the obvious GBSA inputs; keeps geometry | Not clean — pocket geometry still proxies for GBSA inputs | **0.78** |\n> | **3 clean/deployable** | Only 7 RDKit descriptors from SMILES (MW, TPSA, LogP, HBA, HBD, n_heavy, rot_bonds) | **Truly deployable** — no MD run needed | **0.46** |\n>\n> **How to read these numbers:**\n> - r=0.46 = 21 % variance explained (r²). As a pure SMILES scorer, useful for a\n>   **coarse pre-screen** (top 20 % predicted typically covers most true top 40 %),\n>   but not a GBSA replacement.\n> - The collapse **0.93 → 0.78 → 0.46** is the pedagogical core story: define\n>   what \"deployable\" means and enforce it in your feature set.\n>\n> **Model:** `HistGradientBoostingRegressor(max_iter=300, max_depth=4, learning_rate=0.04, l2_regularization=0.5)`\n> — modern gradient-boosted tree with L2 regularisation.\n>\n> **CV:** Leave-One-Target-Out (LOTO), 8 targets (4A5S excluded — no labels).\n> **Metric:** per-target Pearson r, then median across targets, bootstrap 95 % CI (B=1000).\n>\n> **Bonus — Spearman correlation analysis (cells 3-5):** looks for *directional*\n> ligand-chem × GBSA-axis correlations (e.g. \"which ligands prefer intdiel=4?\").\n> BH-FDR correction across 132 tests. No feature survives q<0.10 → still\n> under-powered, but `lig_HBD × intdiel` (ρ=+0.90, uncorrected p=0.005) is suggestive.\n>\n> **Bottom line:** Tier 3 r ≈ 0.46 is the honest number. Not enough to replace\n> GBSA, useful for a fast pre-screen.

In [ ]:
# --- notebook preamble ---
NB_STEM = "41_ligand_chem_gbsa_surrogate"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Follow-up — real ligand chemistry + can we predict GBSA from MD?

Two follow-ups triggered by review of the earlier ligand-chem pass:

1. **Ligand chemistry from topology.** The earlier pass used `ligand_partial_charge_sum`, which is numerical noise (all ligands are net-neutral by construction). We built a proper RDKit descriptor pipeline from `system.top + system.gro` (see `reproduce/ligand_chem_from_topology.py`) that gives 10 real chemistry features per complex — `lig_MW, lig_HBD, lig_HBA, lig_all_rings, lig_LogP, lig_TPSA, lig_fraction_sp3, lig_rot_bonds, lig_n_heavy, lig_partial_q_abs_sum`. We re-ran the per-axis correlation analysis with these features and applied BH-FDR correction across 132 tests (33 features × 4 axes).

2. **Can ML predict per-complex GBSA ΔG from cheap MD features?** If yes, ML becomes a GBSA surrogate — a 1000× cheaper rescoring alternative.

In [ ]:

from scipy.stats import spearmanr
try:
    from statsmodels.stats.multitest import multipletests
except ImportError:
    print('installing statsmodels...')
    import subprocess; subprocess.check_call(['pip','install','-q','statsmodels'])
    from statsmodels.stats.multitest import multipletests

# iter-2 FIX 4c: load_features(with_ligand_chem=True) already includes lig chem,
# so we skip the redundant re-load to avoid _x/_y suffix collisions.
lig = pd.read_parquet(DERIVED / 'ligand_chem.parquet')  # kept for schema reference
# NOTE(fix-pack iter1): removed hardcoded absolute path — GBSA_STUDY comes from discovery9.paths
bt = pd.read_csv(f'{GBSA_STUDY}/data/derived/temporal/bedroc_all_combos_per_target.csv')
full = df.copy()  # df already has ligand-chem columns from load_features(with_ligand_chem=True)
print(f'MD+lig-chem: {len(full)} complexes × {full.shape[1]} cols')

MD_FEATS = [
    'rmsd_bb_mean_A','rmsd_as_bb_mean_A','protein_rg_mean_A','as_ca_rmsf_mean_A',
    'lig_drift_mean_A','lig_com_disp_max_A','lig_escape_frac',
    'lig_internal_rmsd_mean_A','lig_rmsf_mean_A',
    'lig_buried_sasa_mean_A2','vdw_contacts_mean',
    'n_hb_mean','hb_persistence_frac','salt_bridges_lp_mean',
    'ifp_tanimoto_median_vs_ref','lig_binding_modes_2A','lig_orient_autocorr_mean',
    'lig_rg_mean_A','lig_asphericity_mean','lig_dipole_mean_eA',
    'active_site_formal_charge','protein_formal_charge','n_active_site_residues',
]
LIG_FEATS = ['lig_MW','lig_n_heavy','lig_rot_bonds','lig_HBD','lig_HBA','lig_all_rings','lig_LogP','lig_TPSA','lig_fraction_sp3','lig_partial_q_abs_sum']

FP = full.groupby('target')[MD_FEATS + LIG_FEATS].mean()
targets = sorted(set(FP.index) & set(bt.target.unique()))
FP = FP.loc[targets]

def perm_p(rho, x, y, n=5000, seed=0):
    rng = np.random.default_rng(seed); s = abs(rho); ct = 0
    for _ in range(n):
        yp = rng.permutation(y)
        r, _ = spearmanr(x, yp)
        if abs(r) >= s: ct += 1
    return ct/n

axes = {'igb 1→8':('igb',1,8),'intdiel 1→4':('intdiel',1,4),'salt 0→0.15':('saltcon',0.0,0.15),'surften 0→0.0072':('surften',0.0,0.0072)}
rho_mat = pd.DataFrame(index=MD_FEATS+LIG_FEATS, columns=list(axes.keys()), dtype=float)
p_mat = pd.DataFrame(index=MD_FEATS+LIG_FEATS, columns=list(axes.keys()), dtype=float)
for ax_name, (col, lo, hi) in axes.items():
    sub = bt[bt[col].isin([lo, hi])]
    agg = sub.groupby(['target', col]).bedroc20_gbsa.mean().unstack(col)
    delta = (agg[hi] - agg[lo]).reindex(targets)
    for feat in rho_mat.index:
        x = FP[feat].values; y = delta.values
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 5: continue
        rho, _ = spearmanr(x[m], y[m])
        rho_mat.loc[feat, ax_name] = rho
        p_mat.loc[feat, ax_name] = perm_p(rho, x[m], y[m], n=3000)

flat = p_mat.stack().dropna()
_, q, _, _ = multipletests(flat.values, method='fdr_bh')
q_mat = p_mat.astype(float).copy()
for (feat, ax), qv in zip(flat.index, q):
    q_mat.loc[feat, ax] = qv

rows = []
for feat in rho_mat.index:
    for ax in rho_mat.columns:
        rows.append({'feature':feat,'axis':ax,'is_lig':feat in LIG_FEATS,
                     'ρ': rho_mat.loc[feat, ax],'p': p_mat.loc[feat, ax],'q_BH': q_mat.loc[feat, ax]})
rank = pd.DataFrame(rows).dropna(subset=['ρ']).sort_values('q_BH')
print('Top-12 by BH-adjusted q (132 tests):')
print(rank.head(12).round(3).to_string(index=False))
print(f'\nSurvive q<0.10: {int((rank.q_BH < 0.10).sum())} of {len(rank)} tests')

In [ ]:

# Heatmap with lig-chem features marked
LIG_MASK = np.array([f in LIG_FEATS for f in rho_mat.index])
fig, ax = plt.subplots(figsize=(7, max(9, 0.26*len(rho_mat))))
im = ax.imshow(rho_mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(rho_mat.shape[1])); ax.set_xticklabels(rho_mat.columns, rotation=25, ha='right')
labels = [('LIG · ' if L else '     ') + f for f, L in zip(rho_mat.index, LIG_MASK)]
ax.set_yticks(range(rho_mat.shape[0])); ax.set_yticklabels(labels, fontsize=8)
for i in range(rho_mat.shape[0]):
    for j in range(rho_mat.shape[1]):
        r = rho_mat.values[i, j]; qq = q_mat.values[i, j]
        if not np.isfinite(r): continue
        star = ('**' if qq < 0.05 else ('*' if qq < 0.10 else ''))
        col = WHITE if abs(r) > 0.55 else NAVY
        ax.text(j, i, f'{r:+.2f}{star}', ha='center', va='center', fontsize=6.5, color=col)
cbar = plt.colorbar(im, ax=ax, label='Spearman ρ (per-target preference)', shrink=0.6)
cbar.ax.yaxis.label.set_color(NAVY)
ax.set_title('Full correlation matrix (MD + ligand-chem × GBSA axis)\n** = q<0.05, * = q<0.10 after BH-FDR over 132 tests')

**Key finding:** even with real ligand chemistry added, no correlation survives BH-FDR at q < 0.10 (best q = 0.111 for `lig_HBD × intdiel`). With 8 targets and 132 tests, the correction is brutal. Suggestive signals (ρ ≥ 0.83, uncorrected p ≤ 0.02) are physically plausible but need the 18-target validation set to confirm:

- **`lig_HBD` × intdiel:** ρ = +0.90, uncorrected p = 0.004 — ligands with more H-bond donors prefer intdiel=4.
- **`lig_TPSA` × intdiel:** ρ = +0.88, uncorrected p = 0.007 — polar ligands prefer intdiel=4.
- **`lig_LogP` × igb:** ρ = −0.83, uncorrected p = 0.016 — lipophilic ligands prefer igb=1.
- **`protein_formal_charge` × intdiel:** ρ = −0.83, uncorrected p = 0.018.

These track ligand chemistry, which answers "can we predict the best GBSA setting per protein/ligand?" as: directional yes, but the 9-target discovery set is under-powered to lock a rule.

In [ ]:

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut

gbsa_raw = pd.read_csv(f'{GBSA_STUDY}/data/raw/gbsa_dG_raw.csv').rename(columns={'mean_dG_kcalmol':'gbsa_dG'})
LOCKED = 'igb2_di4_salt0.15_st0.0072'
gLK = gbsa_raw[gbsa_raw.combo == LOCKED][['complex_id','target','gbsa_dG']]
data = full.merge(gLK, on=['complex_id','target'], how='inner').dropna(subset=['gbsa_dG'])
print(f'Labeled complexes: {len(data)}   ·   targets: {sorted(data.target.unique())}')

def loto_predict(feature_cols):
    X = data[feature_cols].astype(float).fillna(data[feature_cols].median(numeric_only=True))
    y = data.gbsa_dG.values; g = data.target.values
    logo = LeaveOneGroupOut()
    pred = np.full(len(data), np.nan)
    for tr, te in logo.split(X, y, g):
        m = HistGradientBoostingRegressor(max_iter=300, max_depth=4, learning_rate=0.04, l2_regularization=0.5, random_state=0)
        m.fit(X.iloc[tr], y[tr])
        pred[te] = m.predict(X.iloc[te])
    return pred

pred_md  = loto_predict(MD_FEATS)
pred_lig = loto_predict(LIG_FEATS)
pred_all = loto_predict(MD_FEATS + LIG_FEATS)

# Per-target Pearson r + panel means
tmp = pd.DataFrame({'y': data.gbsa_dG.values, 'MD_only': pred_md, 'lig_only': pred_lig, 'MD_plus_lig': pred_all, 'target': data.target.values})
per_t = tmp.groupby('target').apply(lambda g: pd.Series({
    'MD_only':     np.corrcoef(g.y, g.MD_only)[0,1],
    'lig_only':    np.corrcoef(g.y, g.lig_only)[0,1],
    'MD_plus_lig': np.corrcoef(g.y, g.MD_plus_lig)[0,1],
}), include_groups=False).round(3)
print('\nPer-target Pearson r  (predicting GBSA @ locked combo, LOTO cross-val):')
print(per_t.to_string())
print(f'\nMedian r:   MD_only={per_t.MD_only.median():.3f}  ·  lig_only={per_t.lig_only.median():.3f}  ·  MD+lig={per_t.MD_plus_lig.median():.3f}')

In [ ]:

# Scatter: predicted vs true GBSA for one held-out target
fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=True, sharey=True)
for ax, (t, g) in zip(axes.ravel(), tmp.groupby('target')):
    ax.scatter(g.y, g.MD_only, color=NAVY, s=35, alpha=0.7, label='MD only', edgecolors='none')
    ax.scatter(g.y, g.MD_plus_lig, color=GOLD, s=35, alpha=0.7, label='MD+lig', edgecolors='none')
    lo, hi = g.y.min(), g.y.max()
    ax.plot([lo, hi], [lo, hi], color=GREY, ls=':', lw=1)
    r_md  = np.corrcoef(g.y, g.MD_only)[0,1]
    r_all = np.corrcoef(g.y, g.MD_plus_lig)[0,1]
    ax.set_title(f'{t}   r_MD = {r_md:+.2f}   r_all = {r_all:+.2f}', fontsize=9)
    ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.3)
axes[0,0].legend(fontsize=8, loc='upper left')
fig.suptitle('Predicted vs true GBSA ΔG per held-out target  (LOTO cross-val)',
             y=1.01, color=NAVY, fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('true GBSA ΔG'); axes[1, 0].set_ylabel('predicted GBSA ΔG')
plt.tight_layout()

# Bar chart of per-target r
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(per_t)); w = 0.28
ax.bar(x - w, per_t.MD_only,     w, color=NAVY,  edgecolor=NAVY,   label='MD only')
ax.bar(x,     per_t.lig_only,    w, color=GREYD, edgecolor=NAVY,   label='ligand-chem only')
ax.bar(x + w, per_t.MD_plus_lig, w, color=GOLD,  edgecolor=NAVY,   label='MD + ligand-chem')
ax.axhline(0.5, color=GREY, ls=':', lw=1); ax.axhline(0.9, color=GREY, ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(per_t.index)
ax.set_ylabel('per-target Pearson r  (LOTO)')
ax.set_title('Predicting GBSA ΔG from MD features  ·  panel median MD-only r = {:.2f}'.format(per_t.MD_only.median()))
ax.legend(fontsize=9); ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)

### 17a. Separating quasi-tautology from partial-leak from clean prediction

The `MD_only` feature set in §17 includes `lig_buried_sasa_mean_A2`, `vdw_contacts_mean`, `lig_dipole_mean_eA`, and `salt_bridges_lp_mean` — every one of these is a direct input to the MM-GBSA energy function. So we're asking a regressor to predict a linear combination of quantities it already sees. The r ≈ 0.93 headline is quasi-tautological: it doesn't measure our ability to predict GBSA from a physical MD summary, it measures the regressor's ability to reconstruct a known linear form.

Iter-3 dropped the obvious GBSA-input columns (`sasa|vdw|dipole|charge|coulomb|salt`), leaving MD-stability features (`rmsd_bb_*`, `rmsd_as_bb_*`, `protein_rg_*`, `as_ca_rmsf_*`) in the "honest" set → r ≈ 0.778. But those protein-structure features still partially encode GBSA-relevant pocket geometry (active-site backbone RMSD is a proxy for pocket volume, which GBSA sees). So r=0.778 is a partial-leak estimate, not a clean one.

Iter-4 — the truly leak-free control uses only ligand-chemistry descriptors. These are ligand-only 2D/3D properties, independent of the MD trajectory and therefore of any GBSA input:

* `lig_MW`, `lig_TPSA`, `lig_LogP`, `lig_HBA`, `lig_HBD`, `lig_n_heavy`, `lig_rot_bonds`.

Bootstrap CI on the per-target Pearson r (B=1000, resample targets with replacement, seed 20250901). All three tiers below so you can see the tautology → partial-leak → clean progression. Expected: clean r ≈ 0.455 [0.236, 0.674].

In [ ]:
# iter-4 FIX: LOTO regression control — three tiers of leakage removal, with bootstrap CI.
#   tier 1 (§17): quasi-circular — MD includes GBSA-input columns (sasa/vdw/dipole/charge/coulomb/salt)
#   tier 2 (iter-3, partial-leak): drop GBSA-input columns but keep MD-stability features
#   tier 3 (iter-4, CLEAN): use ONLY ligand-chemistry descriptors (2D/3D, MD-independent)
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut

# Patterns for columns that ARE inputs to the MM-GBSA energy function.
GBSA_INPUT_PATTERNS = ('sasa', 'vdw', 'dipole', 'charge', 'coulomb', 'salt')

def is_gbsa_input(colname: str) -> bool:
    cl = colname.lower()
    return any(p in cl for p in GBSA_INPUT_PATTERNS)

# iter-3 (partial-leak) — MD + LIG minus GBSA-input columns
MD_FEATS_HONEST  = [c for c in MD_FEATS  if not is_gbsa_input(c)]
LIG_FEATS_HONEST = [c for c in LIG_FEATS if not is_gbsa_input(c)]

# iter-4 (CLEAN) — ligand chemistry only. NB: `lig_partial_q_abs_sum` is derived
# from partial charges → it is a GBSA-relevant electrostatic input; exclude it.
LIG_CHEM_ONLY = [c for c in ['lig_MW','lig_TPSA','lig_LogP','lig_HBA','lig_HBD',
                             'lig_n_heavy','lig_rot_bonds'] if c in data.columns]

print(f'MD_FEATS  ({len(MD_FEATS)}): {MD_FEATS}')
print(f'MD_HONEST ({len(MD_FEATS_HONEST)}, GBSA-inputs removed): {MD_FEATS_HONEST}')
print(f'LIG_CHEM_ONLY ({len(LIG_CHEM_ONLY)}, iter-4 clean): {LIG_CHEM_ONLY}')

def loto_predict_cols(feature_cols):
    X = data[feature_cols].astype(float).fillna(data[feature_cols].median(numeric_only=True))
    y = data.gbsa_dG.values; g = data.target.values
    logo = LeaveOneGroupOut()
    pred = np.full(len(data), np.nan)
    for tr, te in logo.split(X, y, g):
        m = HistGradientBoostingRegressor(max_iter=300, max_depth=4, learning_rate=0.04, l2_regularization=0.5, random_state=0)
        m.fit(X.iloc[tr], y[tr])
        pred[te] = m.predict(X.iloc[te])
    return pred

pred_md_h    = loto_predict_cols(MD_FEATS_HONEST)                        # partial-leak
pred_all_h   = loto_predict_cols(MD_FEATS_HONEST + LIG_FEATS_HONEST)     # partial-leak
pred_lig_h   = loto_predict_cols(LIG_FEATS_HONEST) if len(LIG_FEATS_HONEST) else np.full(len(data), np.nan)
pred_ligonly = loto_predict_cols(LIG_CHEM_ONLY)                          # iter-4 CLEAN

honest_tmp = pd.DataFrame({
    'y': data.gbsa_dG.values,
    'MD_honest':          pred_md_h,           # partial-leak (iter-3)
    'lig_honest':         pred_lig_h,          # ligand chemistry (subset of clean)
    'MD_plus_lig_honest': pred_all_h,          # partial-leak (iter-3)
    'lig_chem_only':      pred_ligonly,        # iter-4 CLEAN
    'target': data.target.values,
})
per_t_honest = honest_tmp.groupby('target').apply(lambda g: pd.Series({
    'MD_honest':          np.corrcoef(g.y, g.MD_honest)[0,1],
    'lig_honest':         np.corrcoef(g.y, g.lig_honest)[0,1],
    'MD_plus_lig_honest': np.corrcoef(g.y, g.MD_plus_lig_honest)[0,1],
    'lig_chem_only':      np.corrcoef(g.y, g.lig_chem_only)[0,1],
}), include_groups=False).round(3)

print('\nPer-target Pearson r  (three tiers of leakage removal, LOTO):')
print(per_t_honest.to_string())
median_md_honest    = float(per_t_honest.MD_honest.median())
median_all_honest   = float(per_t_honest.MD_plus_lig_honest.median())
median_ligonly      = float(per_t_honest.lig_chem_only.median())
print(f'\nMedian r:   MD_honest={median_md_honest:.3f}  ·  MD+lig_honest={median_all_honest:.3f}  ·  '
      f'lig_chem_only (CLEAN)={median_ligonly:.3f}')
print(f'For comparison, QUASI-CIRCULAR MD_only median = {per_t.MD_only.median():.3f}')

# Bootstrap CI on the median per-target r, resampling targets with replacement.
RNG_SEED = 20250901
B = 1000
rng = np.random.default_rng(RNG_SEED)
def boot_median_r(series):
    v = np.asarray(series.dropna().values, dtype=float)
    if len(v) < 2:
        return (np.nan, np.nan)
    idx = rng.integers(0, len(v), size=(B, len(v)))
    med = np.median(v[idx], axis=1)
    return float(np.percentile(med, 2.5)), float(np.percentile(med, 97.5))

ci_md_h    = boot_median_r(per_t_honest.MD_honest)
ci_all_h   = boot_median_r(per_t_honest.MD_plus_lig_honest)
ci_ligonly = boot_median_r(per_t_honest.lig_chem_only)
ci_md_orig = boot_median_r(per_t.MD_only)

print(f'\nBootstrap 95% CI on median r (B={B}, resample targets):')
print(f'  (1) MD_only (QUASI-CIRCULAR):           median r = {per_t.MD_only.median():.3f}  CI = [{ci_md_orig[0]:.3f}, {ci_md_orig[1]:.3f}]')
print(f'  (2) MD_honest (PARTIAL-LEAK, iter-3):   median r = {median_md_honest:.3f}  CI = [{ci_md_h[0]:.3f}, {ci_md_h[1]:.3f}]')
print(f'  (3) lig_chem_only (CLEAN, iter-4):      median r = {median_ligonly:.3f}  CI = [{ci_ligonly[0]:.3f}, {ci_ligonly[1]:.3f}]')

# Side-by-side bar chart: three tiers
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4.4))
targets_sorted = list(per_t.index)
x = np.arange(len(targets_sorted)); w = 0.28
r_orig   = per_t.loc[targets_sorted].MD_only.values
r_hon    = per_t_honest.loc[targets_sorted].MD_honest.values
r_clean  = per_t_honest.loc[targets_sorted].lig_chem_only.values
ax.bar(x - w, r_orig,  w, color=NAVY,  edgecolor=NAVY,
       label=f'(1) MD_only quasi-circular (median r={per_t.MD_only.median():.2f})')
ax.bar(x,     r_hon,   w, color=GOLD,  edgecolor=NAVY,
       label=f'(2) MD_honest partial-leak (median r={median_md_honest:.2f})')
ax.bar(x + w, r_clean, w, color=GREYD, edgecolor=NAVY,
       label=f'(3) lig_chem_only CLEAN (median r={median_ligonly:.2f})')
ax.axhline(0.5, color=GREY, ls=':', lw=1); ax.axhline(0.9, color=GREY, ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(targets_sorted)
ax.set_ylabel('per-target Pearson r (LOTO)')
ax.set_title('iter-4 FIX: three tiers of leakage removal in the MD→GBSA surrogate')
ax.legend(fontsize=9); ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()
# (fig auto-captured by preamble hook)


**Verdict — three tiers of leakage separated:**

- **(1) Quasi-circular (§17, includes GBSA-energy inputs):** MD_only Pearson r ≈ 0.93 median across held-out targets. Uses `lig_buried_sasa_mean_A2`, `vdw_contacts_mean`, `lig_dipole_mean_eA`, `salt_bridges_lp_mean` — all direct inputs to MM-GBSA. The regressor is reconstructing a linear combination of quantities it already sees. Not a prediction — a tautology.

- **(2) Partial-leak (iter-3, GBSA-energy inputs removed but MD stability kept):** MD_honest median r ≈ 0.778. Better than random, but the retained protein-structure features (`rmsd_bb_*`, `rmsd_as_bb_*`, `protein_rg_*`, `as_ca_rmsf_*`) still encode GBSA-relevant pocket geometry — active-site backbone RMSD is a proxy for pocket volume. Partial-leak estimate, not clean.

- **(3) Clean (iter-4, only ligand-chemistry descriptors):** lig_chem_only median r ≈ 0.455 [0.236, 0.674]. Truly leak-free — MW, TPSA, LogP, HBA, HBD, n_heavy, rot_bonds are all ligand-2D/3D properties, computed from the topology without any MD trajectory or GBSA input. This is the number to quote when asking "can we predict GBSA per-complex from cheap descriptors?"

**Practical:**
- Do not deploy the MD-surrogate based on §17 alone. The clean r (§17a) is the deployable estimate: r ≈ 0.46 median, CI [0.24, 0.67]. Modest. Needs the 18-target validation set before deployment.
- The gap 0.778 → 0.455 quantifies the residual leakage in the iter-3 "honest" set: roughly a third of what looked like MD-derived predictive signal was actually pocket-geometry proxies for GBSA inputs.
- MD stability + ligand chemistry (honest) is the middle case; use lig_chem_only as the deployment reference and MD_honest as a diagnostic of "how much does MD-observed pocket motion add beyond ligand identity?".

**Caveats:**
- 8 training targets — LOTO r bootstrap CIs are wide.
- Target-family generalisation issues are plausible (see NB 30).
- The 18-target validation set is the honest test. Re-train and re-evaluate before deploying.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
